In [ ]:
import numpy as np
import filters_exp as exp
from matplotlib import pyplot as plt
from numba import njit
from numba_progress import ProgressBar

import metrics
import monte_carlo as mc
import visual_tools as vt
from filters import (
    # parameter dtypes
    std_env_parameters, laplace_env_parameters, sys_inversion_gaussian_env_parameters, 
    sys_inversion_laplacian_env_parameters,
    GMVC_parameters, NLMS_parameters, LMLS_parameters, MCC_parameters, RZA_LMS_parameters,
    sKF_parameters, sKF_L_parameters, AWU_sKF_parameters, AWU_sKF_L_parameters, 
    JO_NLMS_parameters, Shin_VSS_NLMS_parameters, NP_VSS_NLMS_parameters, ENP_VSS_NLMS_parameters,
    RVSS_NLMS_parameters, GISV_sKF_L_parameters,
    # signal / environment helpers
    autocorr_matrix_calc, autocorr_matrix_estimate, AR_settling_time, 
    std_gaussian_behavior, laplace_noise_behavior, sys_inversion_gaussian_behavior, 
    sys_inversion_laplacian_behavior,
    # algorithms
    NLMS_algorithm, GMVC_algorithm, LMLS_algorithm, MCC_algorithm, RZA_LMS_algorithm,
    sKF_algorithm, sKF_L_algorithm, sKF_L_exact_algorithm, sKF_clip_algorithm, 
    JO_NLMS_algorithm, Shin_VSS_NLMS_algorithm, NP_VSS_NLMS_algorithm,
    sKF_integral_algorithm, sKF_L_integral_algorithm,
    AWU_sKF_algorithm, AWU_sKF_L_algorithm, AWU_sKF_algorithm_v2, AWU_sKF_L_algorithm_v2,
    GISV_sKF_algorithm, AWU_sKF_clip_algorithm, ENP_VSS_NLMS_algorithm, RVSS_NLMS_algorithm,
    GISV_sKF_L_algorithm
 )

# Matplotlib configuration
%matplotlib ipympl
%config InlineBackend.figure_format = 'svg'

# General Parameters

In [ ]:
NR = 256
num_workers = 4
num_chunks = 16

N = int(100e3)
L = 16
ho = np.sinc(np.linspace(0,1,L))
ho = ho/np.linalg.norm(ho)
h0 = np.zeros(L)
var_x = 1
var_v = 1e-3 # Gaussian Noise
scale_v = np.sqrt(1e-3/2) # Laplacian Noise

# AR = np.array([1.0, 0.0])
AR = np.array([1.0, -0.6, 0.85])
#AR = np.array([1.0, -0.9, 0.95, -0.8, 0.8])

# Metrics
desired_metrics = {
    'MSD': metrics.squared_deviation,
    'MAD': lambda algorithm_output, noise_signal, true_system: metrics.power_deviation(algorithm_output, noise_signal, true_system, 1),
    'EMSE': metrics.excess_squared_error,
    'EMAE': lambda algorithm_output, noise_signal, true_system: metrics.excess_power_error(algorithm_output, noise_signal, true_system, 1),
    'MSE': metrics.squared_error,
}

plotted_metrics = ['MSD', 'var', 'EMSE']
metrics_scales = {
    'MSD': vt.dB_scale,
    'var': vt.var_scale,
    'EMSE': vt.dB_scale,
}

# Algorithm Parameters
mu = 0.1
delta = 1e-3

epsilon_gaussian = 5e-9
epsilon_laplacian_1 = 6.5e-9
epsilon_laplacian_2 = 2e-8
var_tilde_0 = 1.0 #10000*np.linalg.norm(ho)**2/L

# Time Varying System

## Gaussian noise

In [ ]:
NLMS_Parameters_1 = NLMS_parameters(label="NLMS - mu=1.0", mu=1.0, delta=delta)
NLMS_Parameters_01 = NLMS_parameters(label="NLMS - mu=0.1", mu=0.1, delta=delta)
JO_NLMS_Parameters = JO_NLMS_parameters(label="JO-NLMS", m_0=1e-3, var_w_0=0.0, var_eta=var_v)
sKF_Gaussian_Parameters = sKF_parameters(label="sKF - Gaussian", epsilon=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0)
sKF_Laplacian_Parameters = sKF_L_parameters(label="sKF-L - Laplacian", epsilon=epsilon_laplacian_2, b_eta=scale_v, v_tilde_0=var_tilde_0)
sKF_clip_Parameters = sKF_L_parameters(label="sKF-Clip", epsilon=epsilon_laplacian_1, b_eta=scale_v, v_tilde_0=var_tilde_0)
AWU_sKF_Parameters = AWU_sKF_parameters(label="AWU-sKF - Gaussian", var_eta=var_v, v_tilde_0=var_tilde_0)
GISV_Parameters = sKF_parameters(label="GISV - Gaussian", epsilon=1e3, var_eta=var_v, v_tilde_0=var_tilde_0)
AWU_sKF_L_Parameters = AWU_sKF_L_parameters(label="AWU-sKF-L", b_eta=scale_v, v_tilde_0=var_tilde_0, v_tilde_min=2.5e-6)
AWU_sKF_clip_Parameters = AWU_sKF_L_parameters(label="AWU-sKF-Clip", b_eta=scale_v, v_tilde_0=var_tilde_0, v_tilde_min=2.5e-6)
Shin_VSS_NLMS_Parameters = Shin_VSS_NLMS_parameters(label="Shin-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.01/L, var_e_0=2*var_v/delta)
NP_VSS_NLMS_Parameters = NP_VSS_NLMS_parameters(label="NP-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.05/L, var_e_0=2*var_v, zeta=1e-12)
ENP_VSS_NLMS_Parameters = ENP_VSS_NLMS_parameters(label="ENP-VSS-NLMS", var_eta=var_v, delta=delta, K=2, mu_tau=0.025, var_e_0=2*var_v, zeta=1e-3)
RVSS_NLMS_Parameters = RVSS_NLMS_parameters(label="RVSS-NLMS", epsilon=delta, delta_0=10*var_v/(L*var_x), phi=1-1/L)
GISV_sKF_L_Parameters = GISV_sKF_L_parameters(label="GISV-sKF-L", b_eta=scale_v, v_tilde_0=var_tilde_0, epsilon_min=0.5*epsilon_laplacian_2)


env_parameters = sys_inversion_gaussian_env_parameters(ho=ho, AR=AR, var_v=var_v, var_x=var_x)
environment = sys_inversion_gaussian_behavior

#Algorithms = (NLMS_algorithm, NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, AWU_sKF_algorithm, AWU_sKF_L_algorithm)
#Alg_Parameters = (NLMS_Parameters_1, NLMS_Parameters_01,sKF_Gaussian_Parameters, sKF_Laplacian_Parameters, AWU_sKF_Parameters, AWU_sKF_L_Parameters)
# Algorithms = (NLMS_algorithm, sKF_algorithm, AWU_sKF_algorithm, GISV_sKF_algorithm, JO_NLMS_algorithm, Shin_VSS_NLMS_algorithm)
# Alg_Parameters = (NLMS_Parameters_1, sKF_Gaussian_Parameters, AWU_sKF_Parameters, GISV_Parameters, JO_NLMS_Parameters, Shin_VSS_NLMS_Parameters)
#Algorithms = (NLMS_algorithm, sKF_algorithm, AWU_sKF_algorithm, GISV_sKF_algorithm, Shin_VSS_NLMS_algorithm, NP_VSS_NLMS_algorithm, ENP_VSS_NLMS_algorithm, RVSS_NLMS_algorithm, GISV_sKF_L_algorithm)
#Alg_Parameters = (NLMS_Parameters_1, sKF_Gaussian_Parameters, AWU_sKF_Parameters, GISV_Parameters, Shin_VSS_NLMS_Parameters, NP_VSS_NLMS_Parameters, ENP_VSS_NLMS_Parameters, RVSS_NLMS_Parameters, GISV_sKF_L_Parameters)
Algorithms = (NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, AWU_sKF_algorithm, AWU_sKF_L_algorithm, GISV_sKF_algorithm, GISV_sKF_L_algorithm)
Alg_Parameters = (NLMS_Parameters_1, sKF_Gaussian_Parameters, sKF_Laplacian_Parameters, AWU_sKF_Parameters, AWU_sKF_L_Parameters, GISV_Parameters, GISV_sKF_L_Parameters)

tvs_gaussian_MC_measures, tvs_gaussian_task_tree = mc.dask_MC_Simulations(
    N, NR, env_parameters, environment, Algorithms, Alg_Parameters, desired_metrics, h0, num_workers = num_workers, num_chunks = num_chunks
    )

In [ ]:
fig = vt.plot_metrics(
    tvs_gaussian_MC_measures,
    plotted_metrics,
    metrics_scales=metrics_scales,
    axis_size=(10, 4),
    title = "Gaussian Noise - Time Varying System"
    )

plt.show()

## Laplacian Noise

In [ ]:
NLMS_Parameters_1 = NLMS_parameters(label="NLMS - mu=1.0", mu=1.0, delta=delta)
NLMS_Parameters_01 = NLMS_parameters(label="NLMS - mu=0.1", mu=0.1, delta=delta)
sKF_Gaussian_Parameters = sKF_parameters(label="sKF - Gaussian", epsilon=epsilon_gaussian, var_eta=var_v, v_tilde_0=var_tilde_0)
sKF_Laplacian_Parameters = sKF_L_parameters(label="sKF-L - Laplacian", epsilon=epsilon_laplacian_2, b_eta=scale_v, v_tilde_0=var_tilde_0)
AWU_sKF_Parameters = AWU_sKF_parameters(label="AWU-sKF - Gaussian", var_eta=2*scale_v**2, v_tilde_0=var_tilde_0)
AWU_sKF_L_Parameters = AWU_sKF_L_parameters(label="AWU-sKF-L", b_eta=scale_v, v_tilde_0=var_tilde_0, v_tilde_min=100*epsilon_laplacian_2)
AWU_sKF_clip_Parameters = AWU_sKF_L_parameters(label="AWU-sKF-Clip", b_eta=scale_v, v_tilde_0=var_tilde_0, v_tilde_min=100*epsilon_laplacian_2)
Shin_VSS_NLMS_Parameters = Shin_VSS_NLMS_parameters(label="Shin-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.0065/L, var_e_0=2*var_v/delta)
NP_VSS_NLMS_Parameters = NP_VSS_NLMS_parameters(label="NP-VSS-NLMS", var_eta=var_v, delta=delta, phi=1-0.02/L, var_e_0=2*var_v, zeta=1e-12)
ENP_VSS_NLMS_Parameters = ENP_VSS_NLMS_parameters(label="ENP-VSS-NLMS", var_eta=var_v, delta=delta, K=2, mu_tau=0.01, var_e_0=2*var_v, zeta=1e-3)
GISV_sKF_L_Parameters = GISV_sKF_L_parameters(label="GISV-sKF-L", b_eta=scale_v, v_tilde_0=var_tilde_0, epsilon_min=epsilon_laplacian_1)

env_parameters = sys_inversion_laplacian_env_parameters(ho=ho, AR=AR, scale_v=scale_v, var_x=var_x)
environment = sys_inversion_laplacian_behavior

# Algorithms = (NLMS_algorithm, NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, AWU_sKF_algorithm, AWU_sKF_L_algorithm)
# Alg_Parameters = (NLMS_Parameters_1, NLMS_Parameters_01,sKF_Gaussian_Parameters, sKF_Laplacian_Parameters, AWU_sKF_Parameters, AWU_sKF_L_Parameters)
# Algorithms = (NLMS_algorithm, NLMS_algorithm, sKF_algorithm, AWU_sKF_algorithm, AWU_sKF_algorithm_v2, AWU_sKF_L_algorithm_v2)
# Alg_Parameters = (NLMS_Parameters_1, NLMS_Parameters_01, sKF_Gaussian_Parameters, AWU_sKF_Parameters, AWU_sKF_v2_Parameters, AWU_sKF_L_v2_Parameters)
# Algorithms = (NLMS_algorithm, sKF_algorithm, AWU_sKF_algorithm, GISV_sKF_algorithm, Shin_VSS_NLMS_algorithm, NP_VSS_NLMS_algorithm, ENP_VSS_NLMS_algorithm, RVSS_NLMS_algorithm)
# Alg_Parameters = (NLMS_Parameters_1, sKF_Gaussian_Parameters, AWU_sKF_Parameters, GISV_Parameters, Shin_VSS_NLMS_Parameters, NP_VSS_NLMS_Parameters, ENP_VSS_NLMS_Parameters, RVSS_NLMS_Parameters)
Algorithms = (NLMS_algorithm, sKF_algorithm, sKF_L_algorithm, AWU_sKF_algorithm, AWU_sKF_L_algorithm, GISV_sKF_algorithm, GISV_sKF_L_algorithm)
Alg_Parameters = (NLMS_Parameters_1, sKF_Gaussian_Parameters, sKF_Laplacian_Parameters, AWU_sKF_Parameters, AWU_sKF_L_Parameters, GISV_Parameters, GISV_sKF_L_Parameters)


tvs_laplacian_MC_measures, tvs_laplacian_task_tree = mc.dask_MC_Simulations(
    N, NR, env_parameters, environment, Algorithms, Alg_Parameters, desired_metrics, h0, num_workers = num_workers, num_chunks = num_chunks
    )

In [ ]:
fig = vt.plot_metrics(
    tvs_laplacian_MC_measures,
    plotted_metrics,
    metrics_scales=metrics_scales,
    axis_size=(10, 4),
    title = "Laplacian Noise - Time Varying System"
    )

plt.show()